# QA Source for contraTICO NER Extension

Answers NER entity-aware questions on the 84 source English sentences.
Reuses `qa_entity.py` from biomqm NER extension (source mode).

## Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle')
else:
    print('Running locally')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## Path Configuration

In [ ]:
RESULTS_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline"
EXTENSION_DIR = f"{RESULTS_DIR}/contratico/ner-extension"
BIOMQM_NER_CODE_DIR = f"{RESULTS_DIR}/biomqm/ner-extension/code"

QG_OUTPUT = f"{EXTENSION_DIR}/QG/qg_entity_aware.jsonl"
QA_SOURCE_OUTPUT = f"{EXTENSION_DIR}/QA/source.jsonl"

os.makedirs(f"{EXTENSION_DIR}/QA", exist_ok=True)

if BIOMQM_NER_CODE_DIR not in sys.path:
    sys.path.insert(0, BIOMQM_NER_CODE_DIR)

print(f"QG input: {QG_OUTPUT}")
print(f"QA source output: {QA_SOURCE_OUTPUT}")
print(f"QG exists: {os.path.exists(QG_OUTPUT)}")

## QA Source

Answer entity-aware questions on the 84 source English sentences.

In [ ]:
cmd = [
    sys.executable, "-u",
    f"{BIOMQM_NER_CODE_DIR}/qa_entity.py",
    "--mode", "source",
    "--input_path", QG_OUTPUT,
    "--output_path", QA_SOURCE_OUTPUT
]

print("Running QA Source...")
subprocess.run(cmd, check=True)
print("✓ QA Source complete!")

## Verification

In [ ]:
import json

if os.path.exists(QA_SOURCE_OUTPUT):
    with open(QA_SOURCE_OUTPUT) as f:
        rows = [json.loads(l) for l in f]
    print(f"✓ QA Source: {len(rows)} rows")
    if rows:
        print(f"  Keys: {list(rows[0].keys())}")
        total_a = sum(len(r.get('answers', [])) for r in rows)
        print(f"  Total answers: {total_a}")
else:
    print("✗ QA Source: NOT FOUND")